# 🤖 Sprint 2: Modeling Pipeline (Step 9 s/d Step 13)
### GluCare AI - Prediabetes Clinical Classification Model

Notebook ini mengimplementasikan tahapan pemodelan Machine Learning dari data klinis NHANES:
- **Step 9**: Stratified Train/Test Split (80% Train, 20% Test) & Feature Scaling
- **Step 10**: Pelatihan Model Komparatif (Logistic Regression, Random Forest, Gradient Boosting)
- **Step 11**: Stratified 5-Fold Cross-Validation, Metrik Evaluasi Klinis (Recall, Precision, F1, ROC-AUC)
- **Step 12**: Analisis Feature Importance & Explainability
- **Step 13**: Ekspor Model Pipeline (`glucare_prediabetes_model.joblib`)

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan modul src dapat diimpor
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.model_training import load_data, evaluate_models, train_and_test_best_model

data_path = BASE_DIR / 'data' / 'processed' / 'nhanes_glucare_target_20_40.csv'
models_dir = BASE_DIR / 'models'

X, y, feature_cols = load_data(data_path)
print(f'Fitur Input X ({len(feature_cols)} fitur):', feature_cols)
print(f'Total sampel: {len(X):,} (Bukan Prediabetes: {sum(y==0)}, Prediabetes: {sum(y==1)})')

ModuleNotFoundError: No module named 'numpy'

## Step 9: Stratified Train/Test Split (80:20)
Membagi dataset secara proporsional agar rasio kelas tetap seimbang (62.5% : 37.5%).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training Set: {len(X_train)} sampel')
print(f'Testing Set : {len(X_test)} sampel')

## Step 10 & 11: 5-Fold Stratified Cross-Validation & Model Benchmarking
Membandingkan Logistic Regression, Random Forest, dan Gradient Boosting.

In [ ]:
candidate_models, cv_results = evaluate_models(X_train, y_train, cv_folds=5)
cv_df = pd.DataFrame(cv_results).T
display(cv_df.round(4))

## Step 11: Evaluasi pada Held-out Test Set & Confusion Matrix

In [ ]:
best_name, test_eval = train_and_test_best_model(
    candidate_models, X_train, y_train, X_test, y_test, feature_cols, models_dir
)

## Step 12: Feature Importance Visualization

In [ ]:
import joblib

saved_artifact = joblib.load(models_dir / 'glucare_prediabetes_model.joblib')
pipe = saved_artifact['pipeline']
clf = pipe.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
elif hasattr(clf, 'coef_'):
    imp = np.abs(clf.coef_[0])

feat_df = pd.DataFrame({'feature': feature_cols, 'importance': imp}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_df['feature'], feat_df['importance'], color='#3182CE')
plt.title(f'Feature Importance ({saved_artifact["model_name"]})', fontsize=12, fontweight='bold')
plt.xlabel('Tingkat Kepentingan Relatif')
plt.tight_layout()
plt.show()